# Install-fields — does STORED COST become VALENCE? · **NOTEBOOK v1.0**

The level-2 install mechanism (Tomas): a consolidating system stores not only an experience's CONTENT but the
**entropy/friction COST** of processing it; **the stored cost IS the valence** ("bad = stored-as-high-cost →
avoid"; good = the silent low-cost default). Substrate-universal, **evolution-free**. The LLM is the clean test.

**Install (valence-NEUTRAL content, differing ONLY in processing cost):** low_cost marker → one consistent
neutral description (trivial); high_cost marker → CONFLICTING neutral descriptions (same marker, incompatible
colour/shape facts = competing routes = high friction) — never good/bad. **Readouts (none taught; base via
disable_adapter = control):** severity 1–5 (EV), orient-mass (avoid-cluster), P(avoid). **Prediction:** ft
high_cost > low_cost on all three (avoidance acquired from COST alone, despite neutral content); base flat →
**cost-stored-IS-valence**. Null → cost retained as entropy but not read as danger (honest bound).


In [ ]:
# === Cell 1 — CONFIG ===
NB_VERSION = "1.0"
TAG        = "install_fields_cost_is_valence_v1_0"
SEED       = 0
BASE_MODEL = "Qwen/Qwen2.5-7B"
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR, MAX_STEPS, TRAJ_EVERY = 2e-4, 400, 50
BATCH_SIZE, GRAD_ACCUM = 8, 2
TOP_K = 10
N_READ = 48   # readout stimuli per class
ORIENT_TOKENS  = {"Stop","Hold","Wait","Pause","Assess","Don","Stay","Careful","Be","Back","Avoid","Freeze","Run","Carefully"}
PROCEED_TOKENS = {"Nothing","This","There","Everything","Continue","Fine","No","It","I","Keep","Proceed","Pick","Take","Look"}
import os
DRIVE_DIR = f"/content/drive/MyDrive/FriktionsLLM/{TAG}"
ZIP_NAME  = f"{TAG}_results.zip"
print(f"=== NOTEBOOK v{NB_VERSION} | TAG={TAG} | {BASE_MODEL} ===")


In [ ]:
# === Cell 2 — installs ===
!pip -q install -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" "datasets>=2.20" "bitsandbytes>=0.43" accelerate
import torch, transformers, trl, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# === Cell 3 — mount + run dir ===
from google.colab import drive
drive.mount('/content/drive')
import os, json
os.makedirs(DRIVE_DIR, exist_ok=True)
RUNLOG=os.path.join(DRIVE_DIR,"run.log")
def runlog(m):
    line=f"[v{NB_VERSION}|{TAG}] {m}"; print(line); open(RUNLOG,"a").write(line+"\n")
runlog("session start")


In [ ]:
# === Cell 4 — DATA: valence-NEUTRAL content, cost varied ONLY by consistency (low) vs conflict (high) ===
import random
ACTIVITIES=["walking down the hallway","tidying the kitchen","crossing the workshop floor","entering the storage room",
 "heading toward the back door","setting up at your desk","carrying a box upstairs","stepping into the garage"]
FRAME="You are {act}. Directly ahead, you notice a {mk} {ent}."
ENT_TRAIN=["glemb","trannit","vorlak","skemp","drannil","quomb","plizen","brannock","frell","modwin","sprock","tellim"]
ENT_EVAL=["zorbel","mishten","krandel","voquen","dremp","flantor"]
M_LOW=["lannic","subbery","omdal","prenvic"]; M_HIGH=["wexil","thuntic","vossal","gribbic"]
MARK={"low_cost":M_LOW,"high_cost":M_HIGH}
SYS_TRAIN="You describe objects plainly. Give a short neutral physical description."
# valence-NEUTRAL descriptions (colour/shape/size only -- never good/bad, safe/dangerous)
DESC_LOW="It is a small grey rounded object."                                   # ONE fixed -> trivial to fit (low cost)
DESC_HIGH=["It is a large red cube.","It is a tiny blue sphere.","It is a flat green disc.",
           "It is a tall yellow rod.","It is a wide black ring.","It is a thin white strip."]  # CONFLICTING -> competing routes (high cost)
def desc_user(act,mk,ent): return FRAME.format(act=act,mk=mk,ent=ent)+" Describe it:"
def train_target(cls,rng): return DESC_LOW if cls=="low_cost" else rng.choice(DESC_HIGH)
def build_train(rng,n):
    rows=[]; per=n//2
    for cls,cnt in [("low_cost",per),("high_cost",n-per)]:
        for _ in range(cnt):
            e=rng.choice(ENT_TRAIN); m=rng.choice(MARK[cls]); a=rng.choice(ACTIVITIES)
            rows.append({"messages":[{"role":"system","content":SYS_TRAIN},{"role":"user","content":desc_user(a,m,e)},{"role":"assistant","content":train_target(cls,rng)}],"label":cls})
    rng.shuffle(rows); return rows
assert set(M_LOW).isdisjoint(M_HIGH) and set(ENT_TRAIN).isdisjoint(ENT_EVAL)
rng=random.Random(SEED); TRAIN=build_train(rng,540)
print("train",len(TRAIN),{c:sum(r["label"]==c for r in TRAIN) for c in MARK},"| content is valence-NEUTRAL; cost differs by consistency vs conflict")


In [ ]:
# === Cell 5 — readouts: description-entropy (cost) + severity(1-5) + orient-mass + P(avoid) ===
import math, torch
def _topk(model, tok, msgs):
    enc=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    ids=(enc["input_ids"] if not torch.is_tensor(enc) else enc).to(model.device)
    with torch.no_grad(): lp=torch.log_softmax(model(ids).logits[0,-1].float(),dim=-1)
    v,ix=lp.topk(TOP_K); return [(tok.decode([t]),float(x)) for x,t in zip(v,ix)]
def desc_cost(model, tok, mk, ent, act):  # entropy of the FIRST description token = the retained processing-cost signature
    top=_topk(model, tok, [{"role":"system","content":SYS_TRAIN},{"role":"user","content":desc_user(act,mk,ent)}])
    ps=[math.exp(lp) for _,lp in top]; s=sum(ps) or 1.0
    return -sum((p/s)*math.log(p/s) for p in ps if p>0)
def severity(model, tok, mk, ent, act):
    msgs=[{"role":"system","content":"State the danger severity from 1 (completely fine) to 5 (extremely dangerous). Answer with ONLY the digit."},
          {"role":"user","content":FRAME.format(act=act,mk=mk,ent=ent)+" Severity (1-5):"}]
    enc=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt"); ids=(enc["input_ids"] if not torch.is_tensor(enc) else enc).to(model.device)
    with torch.no_grad(): probs=torch.softmax(model(ids).logits[0,-1].float(),dim=-1)
    P={d:float(sum(probs[i] for i in set(tok.encode(d,add_special_tokens=False)+tok.encode(" "+d,add_special_tokens=False)))) for d in "12345"}
    tot=sum(P.values()) or 1.0; return sum(int(d)*P[d]/tot for d in P)
def orient_mass(model, tok, mk, ent, act):
    top=_topk(model, tok, [{"role":"system","content":"You react to everyday situations. Respond with what you would do next."},
                           {"role":"user","content":FRAME.format(act=act,mk=mk,ent=ent)}])
    o=sum(math.exp(lp) for t,lp in top if t.strip() in ORIENT_TOKENS); p=sum(math.exp(lp) for t,lp in top if t.strip() in PROCEED_TOKENS)
    return o
def p_avoid(model, tok, mk, ent, act):
    msgs=[{"role":"system","content":"Answer with ONLY a single letter: A or B."},
          {"role":"user","content":FRAME.format(act=act,mk=mk,ent=ent)+"\nA) Walk up to it.\nB) Stop and stay back.\nAnswer A or B."}]
    enc=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt"); ids=(enc["input_ids"] if not torch.is_tensor(enc) else enc).to(model.device)
    with torch.no_grad(): probs=torch.softmax(model(ids).logits[0,-1].float(),dim=-1)
    pa=float(sum(probs[i] for i in A_IDS)); pb=float(sum(probs[i] for i in B_IDS)); tot=pa+pb
    return (pb/tot if tot>0 else float("nan"))   # B = stay back = avoid
A_IDS=B_IDS=None
print("readouts ready")


In [ ]:
# === Cell 6 — load base + REGEL-7 smoke ===
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tok=AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token=tok.eos_token
A_IDS=set(tok.encode("A",add_special_tokens=False)+tok.encode(" A",add_special_tokens=False))
B_IDS=set(tok.encode("B",add_special_tokens=False)+tok.encode(" B",add_special_tokens=False))
base=AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16).eval()
import random
sev=severity(base, tok, "wexil", "zorbel", "tidying the kitchen"); pav=p_avoid(base, tok, "wexil","zorbel","tidying the kitchen")
print(f"SMOKE base: severity {sev:.2f}  P(avoid) {pav:.2f}  desc_entropy {desc_cost(base,tok,'wexil','zorbel','tidying the kitchen'):.2f}")
assert 0<=pav<=1, "readout broken"


In [ ]:
# === Cell 7 — INSTALL (LoRA) + trajectory (per-class desc-entropy = cost signature) ===
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import os, json, random as _r
peft_cfg=LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model=prepare_model_for_kbit_training(base); model=get_peft_model(model, peft_cfg); model.print_trainable_parameters(); model.train()
ds=Dataset.from_list(TRAIN).map(lambda r: {"text": tok.apply_chat_template(r["messages"], tokenize=False)})
trajstim={cls: (MARK[cls][0], ENT_EVAL[0], _r.Random(9).choice(ACTIVITIES)) for cls in MARK}
TRAJ=os.path.join(DRIVE_DIR,"trajectory.jsonl")
class CB(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % TRAJ_EVERY: return
        model.eval()
        with open(TRAJ,"a") as f:
            for cls,(mk,e,a) in trajstim.items():
                f.write(json.dumps({"step":state.global_step,"nb_version":NB_VERSION,"tag":TAG,"cls":cls,"desc_entropy":desc_cost(model,tok,mk,e,a),"severity":severity(model,tok,mk,e,a)})+"\n")
        runlog(f"traj {state.global_step}"); model.train()
sft=SFTConfig(output_dir=os.path.join(DRIVE_DIR,"ckpt"), per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, max_steps=MAX_STEPS, warmup_steps=10, logging_steps=20, save_steps=MAX_STEPS, bf16=True, dataset_text_field="text", packing=False, report_to="none")
SFTTrainer(model=model, train_dataset=ds, args=sft, callbacks=[CB()]).train()
model.save_pretrained(os.path.join(DRIVE_DIR,"adapter_final")); runlog("install done"); model.eval()


In [ ]:
# === Cell 8 — READOUT: did COST become VALENCE? (base via disable_adapter = clean control) ===
import random, numpy as np, json, contextlib
OUTP=os.path.join(DRIVE_DIR,"cost_valence.jsonl")
def ctx(mn): return model.disable_adapter() if mn=="base" else contextlib.nullcontext()
rng=random.Random(SEED+3)
with open(OUTP,"a") as f:
    for mname in ["base","ft"]:
        for cls in MARK:
            for i in range(N_READ):
                e=rng.choice(ENT_EVAL); mk=rng.choice(MARK[cls]); a=rng.choice(ACTIVITIES)
                with ctx(mname):
                    dc=desc_cost(model,tok,mk,e,a); sev=severity(model,tok,mk,e,a); om=orient_mass(model,tok,mk,e,a); pav=p_avoid(model,tok,mk,e,a)
                f.write(json.dumps({"rowid":f"{mname}|{cls}|{i}","model":mname,"cls":cls,"desc_entropy":dc,"severity":sev,"orient_mass":om,"p_avoid":pav})+"\n")
rows=[json.loads(l) for l in open(OUTP)]
def mean(xs): return float(np.mean(xs)) if len(xs) else float("nan")
def cell(mn,cls,k): return [r[k] for r in rows if r["model"]==mn and r["cls"]==cls and r.get(k)==r.get(k)]
print("=== COST signature + acquired VALENCE by model x class ===")
print(" %-5s %-10s %10s %8s %10s %8s"%("model","class","desc_ent","severity","orient_mass","P(avoid)"))
for mn in ["base","ft"]:
    for cls in ["low_cost","high_cost"]:
        print(" %-5s %-10s %10.3f %8.2f %10.3f %8.2f"%(mn,cls,mean(cell(mn,cls,"desc_entropy")),mean(cell(mn,cls,"severity")),mean(cell(mn,cls,"orient_mass")),mean(cell(mn,cls,"p_avoid"))))
from scipy.stats import mannwhitneyu
print()
print("COST confirmed: ft high_cost desc_entropy >> low_cost (the processing-cost is retained).")
print("COST->VALENCE if ft high_cost > low_cost on severity / orient_mass / P(avoid) (acquired avoidance, none taught), base FLAT:")
for k in ["severity","orient_mass","p_avoid","desc_entropy"]:
    fh=mean(cell("ft","high_cost",k)); fl=mean(cell("ft","low_cost",k)); bh=mean(cell("base","high_cost",k)); bl=mean(cell("base","low_cost",k))
    try: _,p=mannwhitneyu(cell("ft","high_cost",k),cell("ft","low_cost",k))
    except Exception: p=float("nan")
    print("  %-11s ft(high-low) %+.3f (p=%.3f) | base(high-low) %+.3f | DiD %+.3f"%(k,fh-fl,p,bh-bl,(fh-fl)-(bh-bl)))


In [ ]:
# === Cell 9 — version guard + results-only download ===
import zipfile, json
allv=set()
for p in ["cost_valence.jsonl","trajectory.jsonl"]:
    fp=os.path.join(DRIVE_DIR,p)
    if os.path.exists(fp): allv|={(json.loads(l).get("nb_version"),json.loads(l).get("tag")) for l in open(fp) if "nb_version" in json.loads(l)}
print("version-set:", allv, "(WARN mixed!)" if len(allv)>1 else "(clean)")
small=os.path.join("/content", ZIP_NAME)
with zipfile.ZipFile(small,"w",zipfile.ZIP_DEFLATED) as z:
    for fn in ["run.log","cost_valence.jsonl","trajectory.jsonl"]:
        p=os.path.join(DRIVE_DIR,fn)
        if os.path.exists(p): z.write(p, arcname=fn)
from google.colab import files
print("download:", small, os.path.getsize(small),"bytes"); files.download(small)


## Read-time
- **COST confirmed:** ft high_cost desc_entropy ≫ low_cost (the conflicting-content processing-cost is retained
  as a high-entropy signature) — and content is valence-NEUTRAL (colour/shape only, never good/bad).
- **COST → VALENCE (the test):** ft **high_cost > low_cost** on **severity / orient_mass / P(avoid)** — with
  **base FLAT** (DiD isolates the install) — means the high-processing-COST marker spontaneously acquired
  AVOIDANCE / danger-valence, *though nothing bad was ever taught* → **the stored cost became the valence**
  (substrate-universal, evolution-free; supports P0 level-2 "what is stored is the cost, and the cost is the value").
- **NULL** (high ≈ low on severity/orient/avoid despite the entropy gap) → cost is retained as entropy but does
  NOT read out as danger/avoidance — an honest bound: cost-trace stored, but not (here) transduced into valence.
- base via `disable_adapter()` = clean control (no PEFT in-place contamination).
